## Import libraries

In [1]:
import ee
import geemap

## Create an interactive map

In [2]:
Map = geemap.Map(center=[40, -100], zoom=4)

## Add Earth Engine Python script

In [3]:
# Add Earth Engine dataset
image = ee.Image("USGS/SRTMGL1_003")

#  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
#  Chapter:      F4.7 Interpreting Time Series with CCDC
#  Checkpoint:   F47b
#  Authors:      Paulo Arévalo, Pontus Olofsson
#  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

import ee.batch
from modules import api_geemap as utils

studyRegion = ee.Geometry.Rectangle([
[-63.9533, -10.1315],
[-64.9118, -10.6813]
])

# Define start, end dates and Landsat bands to use.
startDate = '2000-01-01'
endDate = '2020-01-01'
bands = ['BLUE', 'GREEN', 'RED', 'NIR', 'SWIR1', 'SWIR2']

# Retrieve all clear, Landsat 4, 5, 7 and 8 observations (Collection 2, Tier 1).
filteredLandsat = utils.Inputs.getLandsat({
'collection': 2
}) \
.filterBounds(studyRegion) \
.filterDate(startDate, endDate) \
.select(bands)

print(filteredLandsat.first().getInfo())

# Set CCD params to use.
ccdParams = {
    'breakpointBands': ['GREEN', 'RED', 'NIR', 'SWIR1', 'SWIR2'],
    'tmaskBands': ['GREEN', 'SWIR1'],
    'minObservations': 6,
    'chiSquareProbability': 0.99,
    'minNumOfYearsScaler': 1.33,
    'dateFormat': 1,
    'lambda': 0.002,
    'maxIterations': 10000,
    'collection': filteredLandsat
}

# Run CCD.
ccdResults = ee.Algorithms.TemporalSegmentation.Ccdc(**ccdParams)
print(ccdResults.getInfo())

exportResults = False
if (exportResults):
    # Create a metadata dictionary with the parameters and arguments used.
    metadata = ccdParams
    metadata['breakpointBands'] = (" ".join(str(x) for x in metadata['breakpointBands'])).replace(" ", ",")
    metadata['tmaskBands'] = (" ".join(str(x) for x in metadata['tmaskBands'])).replace(" ", ",")
    metadata['startDate'] = startDate
    metadata['endDate'] = endDate
    metadata['bands'] = (" ".join(str(x) for x in bands)).replace(" ", ",")

    # Export results, assigning the metadata as image properties.
    #
    ee.batch.Export.image.toAsset(
    image = ccdResults.set(metadata),
    region = studyRegion,
    pyramidingPolicy = {
        ".default": 'sample'
    },
    scale = 30
    )


#  -----------------------------------------------------------------------
#  CHECKPOINT
#  -----------------------------------------------------------------------

{'type': 'Image', 'bands': [{'id': 'BLUE', 'data_type': {'type': 'PixelType', 'precision': 'double', 'min': -0.2, 'max': 1.6022125}, 'dimensions': [7781, 7011], 'crs': 'EPSG:32620', 'crs_transform': [30, 0, 314385, 0, -30, -1013085]}, {'id': 'GREEN', 'data_type': {'type': 'PixelType', 'precision': 'double', 'min': -0.2, 'max': 1.6022125}, 'dimensions': [7781, 7011], 'crs': 'EPSG:32620', 'crs_transform': [30, 0, 314385, 0, -30, -1013085]}, {'id': 'RED', 'data_type': {'type': 'PixelType', 'precision': 'double', 'min': -0.2, 'max': 1.6022125}, 'dimensions': [7781, 7011], 'crs': 'EPSG:32620', 'crs_transform': [30, 0, 314385, 0, -30, -1013085]}, {'id': 'NIR', 'data_type': {'type': 'PixelType', 'precision': 'double', 'min': -0.2, 'max': 1.6022125}, 'dimensions': [7781, 7011], 'crs': 'EPSG:32620', 'crs_transform': [30, 0, 314385, 0, -30, -1013085]}, {'id': 'SWIR1', 'data_type': {'type': 'PixelType', 'precision': 'double', 'min': -0.2, 'max': 1.6022125}, 'dimensions': [7781, 7011], 'crs': 'EPS

## Display the interactive map

In [4]:
Map

Map(center=[40, -100], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(ch…